In [1]:
# This file explores how Algorithm 4 (MMJ distance by Calculation and Copy) can be accelerated
# by parallel computing, which is discussed in Section 6.2 (Using parallel programming).




In [2]:
 
import time
import pickle
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import pairwise_distances
import networkx as nx
import sys
from joblib import Parallel, delayed
import random
from numba import njit
import threading

In [3]:
 
test_data_145 = pickle.load(  open( "./data/test_data_145.p", "rb" ) ) 

In [4]:
 

@njit(parallel=True, cache=True, fastmath=True)
def primMST_numba(distance_matrix):
    V = distance_matrix.shape[0]
    key = np.full(V, np.inf)
    parent = -np.ones(V, dtype=np.int64)
    inMST = np.zeros(V, dtype=np.bool_)
    key[0] = 0.0
    
    for _ in range(V):
    
        u = -1
        min_val = np.inf
        for v in range(V):
            if (not inMST[v]) and (key[v] < min_val):
                min_val = key[v]
                u = v
        if u == -1:
            break
        inMST[u] = True
        for v in range(V):
        
            if (distance_matrix[u, v] > 0) and (not inMST[v]) and (distance_matrix[u, v] < key[v]):
                key[v] = distance_matrix[u, v]
                parent[v] = u
    return parent

def construct_MST_from_graph(distance_matrix):
    V = distance_matrix.shape[0]

    parent = primMST_numba(distance_matrix)
    
    MST = nx.Graph()
    for i in range(V):
        MST.add_node(i)

    for i in range(1, V):
        MST.add_edge(parent[i], i, weight=distance_matrix[i, parent[i]])
    return MST



def cal_mmj_matrix_by_algo_4_Calculation_and_Copy_parallel_compu(X, round_n=15, n_jobs=-1):
    global mmj_matrix  
    lenX = len(X)
    
    distance_matrix = np.round(pairwise_distances(X), round_n)
    
    mmj_matrix = np.zeros((lenX, lenX)) 
 
   

    MST = construct_MST_from_graph(distance_matrix)
    MST_edge_list = list(MST.edges(data='weight'))

    edge_node_list = [(edge[0], edge[1]) for edge in MST_edge_list]
    edge_weight_list = [edge[2] for edge in MST_edge_list]
    edge_large_to_small_arg = np.argsort(edge_weight_list)[::-1]
    edge_weight_large_to_small = np.sort(edge_weight_list)[::-1]
    edge_nodes_large_to_small = [edge_node_list[i] for i in edge_large_to_small_arg]
    
    MST_list = []
    MST_list.append(MST)
    
    for j in range(num_MST):
        MST_copy = MST.copy(as_view=False)
        MST_list.append(MST_copy)

  
    lock = threading.Lock() 

    N = lenX - 1
    W = min(num_W, N)

    Parallel(n_jobs=n_jobs, backend="threading")(
        delayed(for_parallel_compu)(i, MST_list, edge_nodes_large_to_small, edge_weight_large_to_small, lock)
        for i in range(W)
    )
    
  
    for kk in range(W):
        edge_nodes = edge_nodes_large_to_small[kk]
        MST.remove_edge(*edge_nodes)       
    for i in range(W, N):
        not_for_parallel_compu(i, MST, edge_nodes_large_to_small, edge_weight_large_to_small)
    
 
 
    return mmj_matrix
 


def not_for_parallel_compu(i, MST, edge_nodes_large_to_small, edge_weight_large_to_small):
    global mmj_matrix  
    edge_nodes = edge_nodes_large_to_small[i]
    MST.remove_edge(*edge_nodes)
    
    edge_weight = edge_weight_large_to_small[i]

    tree1_nodes = list(nx.dfs_preorder_nodes(MST, source=edge_nodes[0]))
    tree2_nodes = list(nx.dfs_preorder_nodes(MST, source=edge_nodes[1]))
 
    idx1, idx2 = np.meshgrid(tree1_nodes, tree2_nodes, indexing="ij")
    mmj_matrix[idx1, idx2] = mmj_matrix[idx2, idx1] = edge_weight

 

def for_parallel_compu(i, MST_list, edge_nodes_large_to_small, edge_weight_large_to_small, lock):
    global mmj_matrix
    
    rand_int = random.randint(0, num_MST)
    

    
    MST_temp = MST_list[rand_int]
 

    with lock:
        removed_edges = []
        for kk in range(i + 1):
            edge_nodes = edge_nodes_large_to_small[kk]
            MST_temp.remove_edge(*edge_nodes)
            removed_edges.append(edge_nodes)
        
        edge_weight = edge_weight_large_to_small[i]

        tree1_nodes = list(nx.dfs_preorder_nodes(MST_temp, source=edge_nodes[0]))
        tree2_nodes = list(nx.dfs_preorder_nodes(MST_temp, source=edge_nodes[1]))

    
        for u, v in removed_edges:
            MST_temp.add_edge(u, v, weight=1.0)


    idx1, idx2 = np.meshgrid(tree1_nodes, tree2_nodes, indexing="ij")
    mmj_matrix[idx1, idx2] = mmj_matrix[idx2, idx1] = edge_weight

In [5]:
 
data_id = 136
X = test_data_145[data_id]

print(f"Number of points in data X: {len(X)}" )


num_MST = 1

num_W =  250

mmj_matrix = None

Number of points in data X: 10000


In [6]:
start = time.time()
X_mmj_matrix_algo_4_parallel_compu = cal_mmj_matrix_by_algo_4_Calculation_and_Copy_parallel_compu(X)
end = time.time()
time_used = end - start
time_used = np.round(time_used, 3)

print(f"Time used: {time_used}s" )

Time used: 5.893s
